In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

#Get the URL of the website to scrape
url = "https://books.toscrape.com/"

response = requests.get(url)
soup = BeautifulSoup(response.content, "html.parser")

# Find all the category links on the page
category_links = (soup.find("ul", class_="nav-list").find("ul").find_all("a"))

# Create category dictonary to store category names and their corresponding URLs

categories = {}

for link in category_links:
    category_name = link.get_text(strip=True)
    category_url = url + "/" + link["href"]
    categories[category_name] = category_url

selected_categories = ["Travel", "Mystery", "Historical Fiction"]

#scrape the books from the selected categories

books = []

for category in selected_categories:
    category_url = categories[category]
    response = requests.get(category_url)
    soup = BeautifulSoup(response.content, "html.parser")

    book_list = soup.find_all("article", class_ = "product_pod")

    for book in book_list:
        title = book.h3.a["title"]
        price = book.find("p", class_ = "price_color").get_text(strip=True)
        rating = book.find("p", class_ = "star-rating")["class"][1]
        availability = book.find("p", class_ = "instock availability").get_text(" ", strip=True)

        books.append({
            "Category": category,
            "Title": title,
            "Price": price,
            "Rating": rating,
            "Availability": availability
        })

#Creating a DataFrame from the scraped data

df = pd.DataFrame(books)

#Display the DataFrame

print(df)

print(f"Total Books Scraped: {len(df)}")



# Price
df["price_gdp"] = (
    df["Price"]
    .str.replace("£", "", regex=False)
    .astype(float)
)

# Rating
df["rating"] = df["Rating"].map({
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
})

# Availability
df["in_stock"] = (
    df["Availability"]
    .apply(lambda x: "In stock" in x)
    .astype(int)
)

# GBP → INR
df["price_inr"] = df["price_gdp"] * 105.50





In [ ]:
#Create SQL database

import sqlite3

connection = sqlite3.connect("books_database.db")

cursor = connection.cursor()

#Enable foreign key support
cursor.execute("PRAGMA foreign_keys = ON")

#Create Tables

cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("DROP TABLE IF EXISTS categories")

cursor.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT NOT NULL UNIQUE
)
""")

cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_id INTEGER NOT NULL,
    title TEXT NOT NULL,
    price_gdp REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")

connection.commit()

#Insert categories into the categories table

for category in selected_categories:

    cursor.execute(
        """INSERT INTO categories (category_name) VALUES (?)""", (category,)
    )
connection.commit()

#Read the categories from the database to get their IDs

categories_df = pd.read_sql("SELECT * FROM categories", connection)

print("\n Categories Table:")
print(categories_df)

#Created lookup dictionary

category_lookup = dict(zip(categories_df["category_name"], categories_df["category_id"]))

#Insert books into the books table

books_for_db = df[["Category", "Title", "price_gdp", "rating", "in_stock"]].copy()

# Map the category names to their corresponding IDs

books_for_db["category_id"] = books_for_db["Category"].map(category_lookup)

# Select Database columns

books_for_db = books_for_db[["category_id", "Title", "price_gdp", "rating", "in_stock"]]

# Insert the books into the database

books_for_db.to_sql("books", connection, if_exists="append", index=False)

connection.commit()

print("\n Books successfully inserted.")

In [ ]:
#QUERY 1 SELECT + Where

query1 = """ 
SELECT title, price_gdp, rating FROM books 
WHERE rating >= 4
"""

result1 = pd.read_sql(query1, connection)

print("\n Query 1")
print(result1)

#QUERY 2 ORDER BY + LIMIT

query2 = """
SELECT title, price_gdp, rating FROM books
ORDER BY price_gdp DESC, rating DESC
LIMIT 10
"""

result2 = pd.read_sql(query2, connection)

print("\n Query 2")
print(result2)

#QUERY 3 DISTINCT

query3 = """
SELECT DISTINCT category_id FROM books"""

result3 = pd.read_sql(query3, connection)

print("\n Query 3")
print(result3)

#QUERY 4 IN

query4 = """
SELECT title, price_gdp, rating FROM books
WHERE rating IN (4, 5)"""

result4 = pd.read_sql(query4, connection)

print("\n Query 4")
print(result4)

#QUERY 5 BETWEEN

query5 = """
SELECT title, price_gdp, rating FROM books
WHERE price_gdp BETWEEN 20 AND 30
ORDER BY price_gdp"""

result5 = pd.read_sql(query5, connection)

print("\n Query 5")
print(result5)

#QUERY 6 JOIN

query6 = """
SELECT b.title, b.price_gdp, b.rating, c.category_name, b.in_stock FROM books b
INNER JOIN categories c ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.title ASC
LIMIT 10"""

result6 = pd.read_sql(query6, connection)

print("\n Query 6")
print(result6)


#READ TABLES INTO PANDAS

books_df = pd.read_sql(
    "SELECT * FROM books",
    connection
)

categories_df = pd.read_sql(
    "SELECT * FROM categories",
    connection
)

#REPRODUCE JOIN USING pd.merge()

merge_result = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

#Select exactly the same columns as SQL

merge_result = merge_result[
    [
        "title",
        "price_gdp",
        "rating",
        "category_name",
        "in_stock"
    ]
]

# Same sorting as SQL

merge_result = (
    merge_result
    .sort_values(
        by=["rating", "title"],
        ascending=[False, True]
    )
    .head(10)
    .reset_index(drop=True)
)

print("\nQUERY 6 - pandas.merge()")
print(merge_result)

# Compare SQL JOIN vs PANDAS MERGE

sql_join_result = result6.reset_index(drop=True)

are_equal = sql_join_result.equals(merge_result)

print("\nSQL JOIN and pandas.merge() equivalent:")
print(are_equal)

# SAVE QUERY STRINGS AND OUTPUTS

query_outputs = {
    "Query 1 - SELECT WHERE": (query1, result1),
    "Query 2 - ORDER BY LIMIT": (query2, result2),
    "Query 3 - DISTINCT": (query3, result3),
    "Query 4 - IN": (query4, result4),
    "Query 5 - BETWEEN": (query5, result5),
    "Query 6 - JOIN": (query6, result6)
}

print("\n================ QUERY LOG ================\n")

for query_name, (query, output) in query_outputs.items():

    print(query_name)
    print("------------------------------------------")
    print(query)
    print("OUTPUT:")
    print(output)
    print()

# CLOSE DATABASE

connection.close()

print("Database connection closed.")